<a href="https://colab.research.google.com/github/chandanar8126/Ai_Lab_1/blob/main/Lab5_Adaptive_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-groq chromadb \ sentence-transformers pypdf langchain-huggingface -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/2

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(api_key="API_KEY",

               model="llama-3.1-8b-instant", temperature=0)

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Load the specific PDF you uploaded
loader = PyPDFLoader("Chandana_Dion_Resume.pdf")
docs = loader.load()

# 2. Chop it up into pieces the LLM can quickly search through
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

# 3. Create the vector embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 4. Spin up the 'db' variable globally
db = Chroma.from_documents(documents=splits, embedding=embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
def answer_from_pdf(query):
    # This will now successfully find 'db'
    docs = db.as_retriever().invoke(query)
    context = '\n'.join(d.page_content for d in docs)

    prompt = (
        f"Use ONLY this context to answer.\n{context}\n\n"
        f"Question: {query}\n"
        "If the context does not contain the answer, reply exactly: "
        "'I don't know'."
    )

    return llm.invoke(prompt).content

In [7]:
def adaptive_answer(question, max_tries=3):
    query = question

    for attempt in range(1, max_tries + 1):
        print(f"Attempt {attempt} with query: {query}")

        answer = answer_from_pdf(query)

        # If the LLM successfully answered without saying "I don't know"
        if "i don't know" not in answer.lower():
            return answer  # Return the good answer and stop loop

        # Otherwise, ask the LLM to rephrase the search query for the next loop
        query = llm.invoke(
            f"Rephrase this search query differently to find relevant info: {query}"
        ).content

    return 'Could not find an answer after several tries.'

# Run the final agent test
print(adaptive_answer('What is the main conclusion of the document?'))

Attempt 1 with query: What is the main conclusion of the document?
Attempt 2 with query: Here are a few alternative search queries to find relevant information:

1. "Summary of the document"
2. "Key takeaways from the document"
3. "Main points of the document"
4. "Document abstract or synopsis"
5. "Conclusion or findings of the document"
6. "Document summary or overview"
7. "What is the author's main argument or conclusion?"
8. "Document highlights or main points"
9. "Summary of the document's main points or findings"
10. "Author's conclusion or final thoughts"

You can also try using specific keywords related to the document's topic or subject to refine your search results.

Additionally, you can try searching for:

- Executive summary
- Abstract
- Introduction and conclusion
- Main findings
- Recommendations
- Final thoughts

This will help you find relevant information and get a better understanding of the document's main conclusion.
Based on the provided context, here are some poss